In [4]:
!apt-get update
!apt-get install -y iverilog

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,611 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages 

In [5]:
%%writefile sequence_detector.sv
module sequence_detector (
    input  logic       clk,
    input  logic       reset_n,
    input  logic [2:0] data,
    output logic       sequence_found
);

    typedef enum logic [2:0] {
        IDLE, S1, S2, S3, S4, S5, S6, S7
    } state_t;

    state_t current_state, next_state;

    always_ff @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= IDLE;
        end else begin
            current_state <= next_state;
        end
    end

    always_comb begin
        next_state     = IDLE;
        sequence_found = 1'b0;

        case (current_state)
            IDLE: if (data == 3'b001) next_state = S1;
            S1:   if (data == 3'b101) next_state = S2;
                  else if (data == 3'b001) next_state = S1;
            S2:   if (data == 3'b110) next_state = S3;
                  else if (data == 3'b001) next_state = S1;
            S3:   if (data == 3'b000) next_state = S4;
                  else if (data == 3'b001) next_state = S1;
            S4:   if (data == 3'b110) next_state = S5;
                  else if (data == 3'b001) next_state = S1;
            S5:   if (data == 3'b110) next_state = S6;
                  else if (data == 3'b001) next_state = S1;
            S6:   if (data == 3'b011) next_state = S7;
                  else if (data == 3'b001) next_state = S1;
            S7:   if (data == 3'b101) begin
                      sequence_found = 1'b1;
                      next_state = IDLE;
                  end
                  else if (data == 3'b001) next_state = S1;
            default: next_state = IDLE;
        endcase
    end
endmodule

Overwriting sequence_detector.sv


In [6]:
%%writefile sequence_detector_tb.v
`timescale 1ns/1ps

module tb();
    reg clk;
    reg reset_n;
    reg [2:0] data;
    wire sequence_found;

    integer errors;

    sequence_detector dut (
        .clk(clk),
        .reset_n(reset_n),
        .data(data),
        .sequence_found(sequence_found)
    );

    always begin
        #5 clk = ~clk;
    end

    task apply_stimulus;
        input [2:0] data_value;
        input integer delay_cycles;
        begin
            data <= data_value;
            repeat (delay_cycles) @(posedge clk);
        end
    endtask

    task check_output;
        input integer cycle;
        input expected_value;
        begin
            if (sequence_found !== expected_value) begin
                $display("Error: Cycle %0d, Expected: %b, Got: %b", cycle, expected_value, sequence_found);
                errors = errors + 1;
            end
        end
    endtask

    initial begin
        errors = 0;
        clk <= 0;
        reset_n <= 0;
        data <= 3'b000;

        @(posedge clk);
        reset_n <= 1;

        apply_stimulus(3'b001, 1); check_output(1, 1'b0);
        apply_stimulus(3'b101, 1); check_output(2, 1'b0);
        apply_stimulus(3'b110, 1); check_output(3, 1'b0);
        apply_stimulus(3'b000, 1); check_output(4, 1'b0);
        apply_stimulus(3'b110, 1); check_output(5, 1'b0);
        apply_stimulus(3'b110, 1); check_output(6, 1'b0);
        apply_stimulus(3'b011, 1); check_output(7, 1'b0);
        apply_stimulus(3'b101, 1); check_output(8, 1'b1);

        apply_stimulus(3'b001, 1); check_output(9, 1'b0);
        apply_stimulus(3'b101, 1); check_output(10, 1'b0);
        apply_stimulus(3'b010, 1); check_output(11, 1'b0);
        apply_stimulus(3'b000, 1); check_output(12, 1'b0);

        $display("Mismatches: %d in 12 samples", errors);
        $finish;
    end
endmodule

Overwriting sequence_detector_tb.v


In [7]:
!iverilog -g2012 -o sequence_detector_manual.vvp sequence_detector.sv sequence_detector_tb.v
!vvp sequence_detector_manual.vvp

Mismatches:           0 in 12 samples
